In [1]:
### 1. ### Inizializzazione moduli - eseguire solo una volta
import sys
import json
import logging
from typing import Dict, List, Optional
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from qdrant_client import QdrantClient
from langchain_community.embeddings import FastEmbedEmbeddings

from src.config import (
    QDRANT_URL, 
    COLLECTION_NAME, 
    EMBEDDING_MODEL, 
    RETRIEVAL_TOP_K
)
from src.sidecar_manager import SidecarManager
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget
from src.tag_assigner import TagAssigner

# Configurazione del formato dei log per il notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Inizializzazione Vector Store con FastEmbed nativo
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL)

# Per il DataSet Pre-Taggato
sidecar_path = project_root / "data/processed/armstrong/sc_multi_pret_distanceclasses1.json"

# DataSet privo di Tag
#sidecar_path = project_root / "data/processed/armstrong/sidecar_nopretag3.json"

# Istanziazione unica sidecar. Eseguire solo una volta per evitare il warning
sidecar_manager = SidecarManager(filepath=sidecar_path)

# Inizializzazione widget
widget = ChunkGraphWidget(sidecar=sidecar_manager)

# Inizializzazione moduli gestinoe Tag
tag_assigner = TagAssigner()

print("!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita")

/tmp/ipykernel_9296/2443327220.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings
/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


!>> Moduli caricati e connessione a Qdrant/FastEmbed stabilita


In [2]:
### 2. ### Scelta conversation_tag
conversation_tags = ["Jazz"]
print(f"!>> La conversazione parla di: {conversation_tags}")

!>> La conversazione parla di: ['Jazz']


In [3]:
### 3.OPTION_A ### Selezione query da eval_queries
queries_path = project_root / "data/queries/armstrong/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

#Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
matched = [q for q in benchmark_queries if q.get("id") == "Q_SPEC_ASTRO_01"]
query_obj = matched[0]
query_text = query_obj["query"]

print(f"  ?>> ID Query: {query_obj['id']}")

  ?>> ID Query: Q_SPEC_ASTRO_01


In [ ]:
### 3.OPTION_B ### Digitazione query
query_text = ""

In [4]:
### 4.1 ### Query selezionata e Tag globali disponibili
sidecar_data = sidecar_manager.load_data()
global_tags_dict = sidecar_data.get("global_tags", {})
global_tags = list(global_tags_dict.keys())

query_vector = embeddings.embed_query(query_text)
print(f"!>> Testo Query: '{query_text}'\n")

print(f"!>> Tag Globali: {global_tags}")

!>> Testo Query: 'Quale ruolo ha avuto Armstrong nella missione Apollo 11 e quali frasi celebri ha pronunciato?'

!>> Tag Globali: ['Jazz', 'Ciclismo', 'Astronautica']


In [1]:
### 4.2 ### Assegnazione TAG alla query

# metodo per costruzione dei query_tags
def build_query_tags(
    conversation_tags: Optional[List[str]] = None,
    assigned_tags: Optional[List[str]] = None,
    automatic_tags: Optional[List[str]] = None,
) -> Dict[str, List[str]]:
  """Costruisce il dizionario query_tags garantendo la disgiunzione gerarchica dei tag:
  conversation_tags > assigned_tags > automatic_tags.
  """
  seen = set()

  clean_conv: List[str] = []
  for t in conversation_tags or []:
    if t and t not in seen:
      clean_conv.append(t)
      seen.add(t)

  clean_assigned: List[str] = []
  for t in assigned_tags or []:
    if t and t not in seen:
      clean_assigned.append(t)
      seen.add(t)

  clean_auto: List[str] = []
  for t in automatic_tags or []:
    if t and t not in seen:
      clean_auto.append(t)
      seen.add(t)

  all_tags: List[str] = clean_conv + clean_assigned + clean_auto

  return {
      "conversation_tags": clean_conv,
      "assigned_tags": clean_assigned,
      "automatic_tags": clean_auto,
      "all_tags": all_tags,
  }

# Assegnazione
assigned_tags = ["Astronautica"]

auto_tag_assignation = True

automatic_tags = []
if global_tags and auto_tag_assignation:
    automatic_tags = tag_assigner.assign_tags(
        text=query_text, candidate_tags=global_tags, vector=query_vector
    )

# Unisce tramite il metodo apposito
query_tags = build_query_tags(
    conversation_tags=conversation_tags,
    assigned_tags=assigned_tags,
    automatic_tags=automatic_tags,
)

print(f"!>> Tag assegnati alla query:")
print(f"   >>> Tag di Conversazione: {conversation_tags}")
print(f"   >>> Tag Manuali della Query: {assigned_tags}")
if auto_tag_assignation:
    if global_tags:
        print(f"   >>> Tag Automaticamente assegnati alla Query: {automatic_tags}")
        print(f"       |> Tag Globali disponibili: {global_tags}")
    else:
        print("   >>> Assegnazione Automatica abilitata, ma non sono presenti Tag Globali da assegnare")
else:
    print("   >>> Assegnazione Tag Automatici disabilitata")
print(f"<<< Lista completa: {all_tags}")

NameError: name 'global_tags' is not defined

In [7]:
### 5 ### Retrieval vettoriale - ("1° retrieval")
ARMSTRONG_TOP_K = 5

response = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=ARMSTRONG_TOP_K,
    with_vectors=True,
    with_payload=True
)
raw_records = response.points

print(f"!>> Recuperati {len(raw_records)} record con relativi vettori di embedding")

!>> Recuperati 5 record con relativi vettori di embedding


In [10]:
### 6 ### Visualizzazione grafo chunk dal 1° retrieval
# Grafo Retrieved
builder = KnowledgeGraphBuilder()
tag_overrides = sidecar_manager.load_data().get("tag_overrides", {})

for idx, rec in enumerate(raw_records):
    payload = rec.payload or {}
    
    # Costruzione dell'identificativo unico del chunk
    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload

    doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"

    raw_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
    chunk_idx = raw_idx if raw_idx is not None else idx_global

    chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"
    
    # Estrazione testo
    text = payload.get("text", payload.get("page_content", ""))

    # Estrazione tag - da la priorità a quelli del sidecar, o quelli salvati su Qdrant (in teoria non presenti)
    base_tags = payload.get("user_tags") or meta.get("user_tags") or payload.get("tags") or meta.get("tags") or []
    chunk_sidecar_info = tag_overrides.get(chunk_id, {})
    tags = chunk_sidecar_info.get("user_tags", base_tags)
    
    builder.add_chunk_node(
        chunk_id=chunk_id,
        text=text,
        vector=rec.vector,
        tags=tags
    )

builder.auto_connect_nodes()

graph_data = builder.to_json_data()

print(f"!>> Grafo generato con successo:")
print(f"   >>> Nodi inseriti: {len(graph_data['nodes'])}")
print(f"   >>> Archi collegati: {len(graph_data['links'])}")

### Inizializzazione e rendering del Widget
widget.load_graph(graph_data)

widget

!>> Grafo generato con successo:
   >>> Nodi inseriti: 5
   >>> Archi collegati: 10


In [ ]:
7-> reranker TOP_N: output reranked_results_top_n, dist_retrieved_top_n
8-> plot di raw_records+dist_retrieved_top_n per grafo di retrieved e vicini
    plot di reranked_results_top_n
9-> reranker TOP_K: output reranked_results_top_k, dist_retrieved_top_k
10-> plot di raw_records+dist_retrieved_top_k per grafo di retrieved e vicini
     plot di reranked_results_top_k
11-> eventuali ulteriori prove con altri valori di top_j utilizzando il reranker generico

In [ ]:
### 7 ### Retrieval su Classi di Distanza - ("2° retrieval")


NOTA PRE_IMPLEMENTATIVA:
>>>A POSTERIORI quanto scritto di seguito può essere in realtà un ragionamento più giusto da fare per i rereanker
   invece che per il reretriever, che semplicemetne prende tutta la lista (non ordinata) degli adiacenti.
   Poi sta al reranker fare tutte le considerazioni seguenti, lasciando il reretriever come un semplicemente cieco lavoratore.
   L'alternativa è che il reretrieval selezioni già di suo i migliori TOP_QUALCOSA per evitare di passare una lista potenzialmente
   lunga n
Va scelto se la questione di vedere quanti dei chunk originali hanno il conv_tag si fa in un reranker apposito che quindi
verrebbe fatto prima (e divida i chunk in 2 categorie: CON e SENZA conv_tag).
Poi va scelto se fare il 2° retrieval solo se TOP_K-CON_CONV_TAG = SENZA_CONV_TAG > 0 o anche se = 0;
cioè hanno tutti il conv_tag ma si vuole comunque provare a dare una speranza ai vicini.
--->Questo potrebbe essere un parametro preso in input dai metodi: tipo GET_IF_ALL_WITHCONVTAG=BOOLEAN
    e un secondo parametro HOW_MANY_MORE_TO_GET che indica appunto quanti ancora prenderne. Similmente si potrebbe utilizzare
    questo paramtro HOW_MANY_MORE_TO_GET anche nel caso in cui SENZA_CONV_TAG>0, sempre per indicare quanti in più ne voglio aggiungere

NOTA IMPLEMENTATIVA:
metodi da una classe distance_retriever:

----> unisci i 2 seguenti metodi in uno solo, che si differenziano in uno o nell'altro a seconda di un valore BOOLEANO
      passato in input nella firma del metodo
>metodo get_AVVICINATIORMORE_chunks()
dà la lista non ordinata dei nodi dei chunk retrieved al 1°
+ gli adiacenti avvicinati o molto avvicinati (returnando anche la classe a cui appartengono)
ciascun chunk viene anche salvato con il punteggio orginale e categorizzato in una delle 4 categorie (vedi note)
>metofo get_ALLADJ_nodes()
restituise anche i nodi allontanati o molto allontanati




In [ ]:
### 8 ### Grafo chunk post 2° retrieval (anche con nodi vicini)

fa vedere sia i TOP_K nodi originariamente retrieved che tutti i vicini di
quei K nodi che sono stati presi dal get_...() precedente.
E' un grafo quindi molto più ampio



In [ ]:
### 9.OPT_A ### Reranking: 1° metodo (TOP_N)

altra classe che lavora in combo con il distance_retriever:
prende l'oggetto returnato dai get_...() di distance_retriever
e lo riordina secondo i criteri esposti su note nel 1° metodo
Prende un parametro CUT_INDEX a cui si passa quanti nodi si vuole tenere
nella lista che saranno returnati

Usa il metodo sopra, tagliando al TOP_N (cut_index=TOP_N)



In [ ]:
### 9.OPT_B ### Reranking: 2° metodo (TOP_K)

Come sopra, ma tagliando al TOP_K (cut_index=TOP_K)



In [ ]:
### 9.OPT_C ### Reranking: 3° metodo (ALL)

Metodo potenzialmente pesante e probabilmente non valido, ma implementato come prova: si passano tutti i nodi "AVVICINATI"
o "MOLTO AVVICINATI" che il re-retriever ha preso e il reranking ha ordinato come scritto su note.
Così si vede come opera un LLM con un contesto potenzialmente enorme

In [ ]:
### 10 ### Grafo chunk post-reranking



